In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Step 1: Load and preprocess data
file_path = '<EXTERNAL_DATA_ROOT>'
excel_data = pd.ExcelFile(file_path)

# Combine all summary sheets into a dictionary for substrates and products
summary_sheets = {sheet: pd.read_excel(file_path, sheet_name=sheet) for sheet in excel_data.sheet_names if "_summary" in sheet}

# Extract unique substrates and products
all_substrates = set()
all_products = set()
for sheet_data in summary_sheets.values():
    all_substrates.update(sheet_data['substrates'].dropna().unique())
    all_products.update(sheet_data['products'].dropna().unique())

# Add additional products
additional_products = ["Histamine", "Glunolactone", "4-ACETAMIDOBUTANOATE", "N,N,N-TRIMETHYLLYSINE"]
all_products.update(additional_products)

# Sort substrates and products
all_substrates = sorted(all_substrates)
all_products = sorted(all_products)

# Create binary matrices
substrate_matrix = np.zeros((len(all_substrates), len(summary_sheets)))
product_matrix = np.zeros((len(all_products), len(summary_sheets)))

# Fill binary matrices
species_names = list(summary_sheets.keys())
for j, (sheet_name, sheet_data) in enumerate(summary_sheets.items()):
    for i, substrate in enumerate(all_substrates):
        if substrate in sheet_data['substrates'].values:
            substrate_matrix[i, j] = 1
    for i, product in enumerate(all_products):
        if product in sheet_data['products'].values:
            product_matrix[i, j] = 1

# Replace species names to remove "_summary" suffix
species_names_cleaned = [name.replace("_summary", "") for name in species_names]

# Load mappings for Names and Classes
substrate_names_sheet = pd.read_excel(file_path, sheet_name='SubstrateNames')
product_names_sheet = pd.read_excel(file_path, sheet_name='ProductNames')

substrate_mapping = substrate_names_sheet.set_index("KEGG ID")["Names"].to_dict()
product_mapping = product_names_sheet.set_index("KEGG ID")["Name"].to_dict()

substrate_classes = substrate_names_sheet.set_index("Names")["Class"].to_dict()
product_classes = product_names_sheet.set_index("Name")["Class"].to_dict()

# Replace IDs with Names and group by Class
all_substrates = [substrate_mapping.get(kegg, kegg) for kegg in all_substrates]
all_products = [product_mapping.get(kegg, kegg) for kegg in all_products]

substrates_with_classes = sorted(zip(all_substrates, [substrate_classes.get(name, "Unknown") for name in all_substrates]), key=lambda x: x[1])
products_with_classes = sorted(zip(all_products, [product_classes.get(name, "Unknown") for name in all_products]), key=lambda x: x[1])

all_substrates, substrate_classes_sorted = zip(*substrates_with_classes)
all_products, product_classes_sorted = zip(*products_with_classes)

# Combine matrices
combined_matrix = np.vstack((substrate_matrix, product_matrix))
combined_labels = list(all_substrates) + list(all_products)

# Step 2: Plot Nature-style heatmap
fig, ax = plt.subplots(figsize=(14, 18))

# Define colors
substrate_color = "#377eb8"  # Muted blue
product_color = "#4daf4a"    # Muted green
divider_color = "#e41a1c"    # Muted red

# Plot substrates
for i, j in zip(*np.where(substrate_matrix == 1)):
    ax.scatter(j, i, marker='s', color=substrate_color, s=100, edgecolor="black", label='Substrate' if i == 0 and j == 0 else "")

# Plot products
for i, j in zip(*np.where(product_matrix == 1)):
    ax.scatter(j, i + len(all_substrates), marker='o', color=product_color, s=100, edgecolor="black", label='Product' if i == 0 and j == 0 else "")

# Add divider
ax.axhline(y=len(all_substrates) - 0.5, color=divider_color, linewidth=2.5, linestyle="--", label="Substrate-Product Divider")

# Add row shading
for i in range(len(all_substrates)):
    if i % 2 == 0:
        ax.axhspan(i - 0.5, i + 0.5, color="#f7f7f7", alpha=0.5, lw=0)
for i in range(len(all_products)):
    if i % 2 == 0:
        ax.axhspan(i + len(all_substrates) - 0.5, i + len(all_substrates) + 0.5, color="#f7f7f7", alpha=0.5, lw=0)

# Customize labels and title
ax.set_title("Nature-Style Heatmap: Substrate and Product Secretion", fontsize=22, fontweight='bold', color="black")
ax.set_xticks(range(len(species_names_cleaned)))
ax.set_xticklabels(species_names_cleaned, rotation=90, fontsize=10, fontweight='bold', color="black")
ax.set_yticks(range(len(combined_labels)))
ax.set_yticklabels(combined_labels, fontsize=9, color="black")

# Enhance legend
handles, labels = ax.get_legend_handles_labels()
legend = ax.legend(handles[:3], ['Substrate', 'Product', 'Divider'], loc='upper left', fontsize=12, frameon=True, fancybox=True, shadow=False)
legend.get_frame().set_edgecolor('black')
legend.get_frame().set_linewidth(1.5)

# Add gridlines
ax.grid(visible=True, which='major', color='lightgrey', linestyle=':', linewidth=0.8)
ax.set_axisbelow(True)

# Tight layout
plt.tight_layout()

# Save the final plot
nature_style_heatmap_path = "ExperimentalMetabolomics.pdf"
plt.savefig(nature_style_heatmap_path, dpi=300)
plt.show()

print(f"Saved heatmap: {nature_style_heatmap_path}")
